In [1]:
# creating spark session
exec(open('/home/jovyan/.ipython/profile_default/startup/00-spark-session.py').read())

Spark 3.5.0 session ready as `spark` (Delta Lake enabled).


# Report Contiguous Dates

## Difficulty
Medium

## Topics
- PySpark
- SQL
- Window Functions
- Date Functions
- Gaps and Islands
- Row Number
- Group By
- Aggregation

## Problem Statement

You are given a PySpark DataFrame named `tasks` containing task execution information.

### Dataset: `tasks`

| Column | Data Type | Description |
|---|---|---|
| `task_id` | Integer | Unique identifier for each task |
| `status` | String | Task status: either `succeeded` or `failed` |
| `task_date` | String | Date on which the task was executed in `YYYY-MM-DD` format |

## Task

Find all **contiguous date ranges** where tasks have the same status.

A contiguous range is a sequence of **consecutive calendar dates with no gaps** where all dates have the same status.

### Requirements

1. Order the tasks by `task_date`.
2. Identify consecutive dates that have the same `status`.
3. Create a separate group for each contiguous range.
4. For each range, determine:
   - `start_date` → earliest date in the range
   - `end_date` → latest date in the range
5. A single date without an adjacent date having the same status should be treated as its own range.
6. Return:
   - `status`
   - `start_date`
   - `end_date`
7. Sort the final result by `start_date` in ascending order.

## Example

The task statuses are:

- January 1 → succeeded
- January 2 → succeeded
- January 3 → succeeded
- January 4 → failed
- January 5 → failed
- January 6 → succeeded
- January 7 → succeeded

The contiguous ranges are:

| status | start_date | end_date |
|---|---|---|
| succeeded | 2023-01-01 | 2023-01-03 |
| failed | 2023-01-04 | 2023-01-05 |
| succeeded | 2023-01-06 | 2023-01-07 |

## Expected Output

| status | start_date | end_date |
|---|---|---|
| succeeded | 2023-01-01 | 2023-01-03 |
| failed | 2023-01-04 | 2023-01-05 |
| succeeded | 2023-01-06 | 2023-01-07 |

In [2]:
    from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType
)

tasks_data = [
    (1, "succeeded", "2023-01-01"),
    (2, "succeeded", "2023-01-02"),
    (3, "succeeded", "2023-01-03"),
    (4, "failed",    "2023-01-04"),
    (5, "failed",    "2023-01-05"),
    (6, "succeeded", "2023-01-06"),
    (7, "succeeded", "2023-01-07")
]

tasks_schema = StructType([
    StructField("task_id", IntegerType(), False),
    StructField("status", StringType(), False),
    StructField("task_date", StringType(), False)
])

tasks = spark.createDataFrame(
    tasks_data,
    tasks_schema
)

tasks.show()
tasks.printSchema()

+-------+---------+----------+
|task_id|   status| task_date|
+-------+---------+----------+
|      1|succeeded|2023-01-01|
|      2|succeeded|2023-01-02|
|      3|succeeded|2023-01-03|
|      4|   failed|2023-01-04|
|      5|   failed|2023-01-05|
|      6|succeeded|2023-01-06|
|      7|succeeded|2023-01-07|
+-------+---------+----------+

root
 |-- task_id: integer (nullable = false)
 |-- status: string (nullable = false)
 |-- task_date: string (nullable = false)



# Using SQL

In [3]:
tasks.createOrReplaceTempView("tasks")

In [42]:
spark.sql("""
    with cte1 AS 
    (
        SELECT task_id,
        status,task_date,
        row_number() OVER( order by task_date) as rn_all, 
        row_number() over(partition by status order by task_date) as rn_status 
    from tasks
    ),
    
    cte2 AS (
        SELECT status,
        task_date,
        rn_all - rn_status AS grp 
    from cte1
        
    )
    
    SELECT status,
        min(task_date) as start_date,
        max(task_date) as end_date
        from cte2
        group by grp,status 
        ORDER by start_date 
    
""").show()

+---------+----------+----------+
|   status|start_date|  end_date|
+---------+----------+----------+
|succeeded|2023-01-01|2023-01-03|
|   failed|2023-01-04|2023-01-05|
|succeeded|2023-01-06|2023-01-07|
+---------+----------+----------+

